# Конспект. Модуль 3: Матчасть бустинга — функция потерь и градиентный спуск в пространстве функций

## 1. Зачем это нужно и как это связано с предыдущими модулями

В Модуле 2 мы выяснили: Random Forest прекрасно **гасит variance** усреднением независимых деревьев, но почти не трогает **bias** каждого отдельного дерева. Если после усреднения множества глубоких деревьев модель всё ещё систематически ошибается (например, плохо ловит редкие, но специфичные паттерны мошенничества) — это уже проблема bias, и добавление ещё большего числа независимых деревьев её не решит: формула `Var(среднее) = ρσ² + (1-ρ)σ²/B` в разделе о bagging **вообще не содержит bias** — усреднение принципиально не может его снизить.

Нужен другой механизм — **последовательная коррекция ошибок**: строим модель, смотрим, где она ошибается, строим следующую модель специально для исправления этих ошибок, и так далее. Это идея бустинга. Но чтобы формализовать её математически строго (а не на уровне «дерево учится на остатках предыдущего»), нужен небольшой, но важный кусок матаппарата — этому и посвящён Модуль 3. **В этом модуле мы не строим ни одного дерева** — только разбираемся с математикой, которая объясняет, *что именно* должно предсказывать следующее дерево. Само дерево появится в Модуле 4.

## 2. Напоминание: что вообще значит «обучить модель»

Формально, обучение почти любой модели — это задача оптимизации: найти такие параметры (или в общем случае — такую функцию) `F`, которая минимизирует **функцию потерь** `L`, усреднённую по обучающей выборке:

In [ ]:
F* = argmin(F)  (1/N) · Σ(i=1..N) l(y_i, F(x_i))

где `l(y_i, F(x_i))` — «штраф» за расхождение между истинным значением `y_i` и предсказанием модели `F(x_i)` для одного конкретного объекта. Вы уже встречали конкретные примеры `l`:
- **MSE** (регрессия): `l(y, F) = (y - F)²` (или `(1/2)(y-F)²` — множитель `1/2` берут исключительно для удобства при взятии производной, на результат оптимизации он не влияет).
- **LogLoss** (классификация, Неделя 4): `l(y, F) = -[y·log(p) + (1-y)·log(1-p)]`, где `p` — предсказанная вероятность положительного класса.

Вопрос, на который отвечает этот модуль: **как именно найти такую `F`, если `F` — это не пара чисел (весов), а произвольная сложная функция (в перспективе — целый ансамбль деревьев)?**

## 3. Напоминание: обычный градиентный спуск (в пространстве параметров)

Прежде чем перейти к «пространству функций», зафиксируем то, что должно быть интуитивно понятно из базового матанализа.

**Одна переменная.** Пусть у нас есть функция `f(θ)`, и мы хотим найти `θ`, минимизирующее `f`. Производная `f'(θ)` показывает **направление и скорость роста** функции в точке `θ`. Если `f'(θ) > 0`, функция растёт при увеличении `θ` — значит, чтобы **уменьшить** `f`, нужно двигаться в сторону **уменьшения** `θ`, то есть в сторону, противоположную производной.

**Правило обновления:** `θ_new = θ_old - η · f'(θ_old)`, где `η` (learning rate) — размер шага.

**Простой числовой пример.** Пусть `f(θ) = (θ - 3)²` (минимум очевидно в `θ=3`, значение 0). Производная: `f'(θ) = 2(θ-3)`.

Начнём с `θ_0 = 0`, `η = 0.1`:
- `f'(0) = 2·(0-3) = -6` -> `θ_1 = 0 - 0.1·(-6) = 0.6`
- `f'(0.6) = 2·(0.6-3) = -4.8` -> `θ_2 = 0.6 - 0.1·(-4.8) = 1.08`
- `f'(1.08) = 2·(1.08-3) = -3.84` -> `θ_3 = 1.08 - 0.1·(-3.84) = 1.464`

Видно, что `θ` монотонно приближается к 3, шаг за шагом — это и есть градиентный спуск в действии на игрушечном примере.

**Много переменных.** Если `f` зависит от вектора параметров `θ = (θ_1, ..., θ_k)`, то вместо одной производной берём **вектор частных производных** — **градиент**:

In [ ]:
∇f(θ) = ( ∂f/∂θ_1, ∂f/∂θ_2, ..., ∂f/∂θ_k )

Каждая частная производная `∂f/∂θ_j` показывает, как меняется `f`, если менять **только** `θ_j`, оставляя все остальные координаты фиксированными. Обновление то же самое, но покомпонентно (или в векторном виде): `θ_new = θ_old - η·∇f(θ_old)`.

**Почему движение против градиента гарантированно уменьшает функцию (при малом `η`)?** Разложение Тейлора первого порядка рядом с точкой `θ`:

In [ ]:
f(θ - η∇f(θ)) ≈ f(θ) - η · ||∇f(θ)||²

Так как `||∇f(θ)||² ≥ 0` (сумма квадратов всегда неотрицательна), а `η > 0`, второе слагаемое **вычитается** — то есть при достаточно малом `η` новое значение функции гарантированно **не больше** старого. Это и есть формальное обоснование того, почему антиградиент — направление наискорейшего локального убывания.

## 4. Ключевой концептуальный скачок: градиентный спуск в пространстве функций

Вот центральная идея всего модуля, к которой готовила вся предыдущая секция.

**В обычном градиентном спуске** мы оптимизируем **фиксированное число параметров** (например, веса линейной регрессии — их, скажем, 10 или 20, но заранее известно сколько). Мы двигаем сам вектор весов `θ` шаг за шагом.

**В градиентном бустинге мы поступаем принципиально иначе.** Представим себе на секунду, что у нас **нет** параметрической модели вообще, а есть просто **вектор предсказаний** для всех `N` обучающих объектов:

In [ ]:
F = ( F(x_1), F(x_2), ..., F(x_N) )

Этот вектор из `N` чисел (по одному на каждый обучающий объект) можно **временно считать** «параметрами», которые мы хотим оптимизировать — то есть перейти от бесконечномерного пространства «всех возможных функций `F: X->ℝ`» к его конкретному срезу на обучающих точках: `N`-мерному пространству возможных наборов предсказаний.

Функция потерь как функция этого вектора:

In [ ]:
L(F) = (1/N) · Σ(i=1..N) l(y_i, F(x_i))

Так как каждое слагаемое суммы зависит **только от одной координаты** `F(x_i)` (и ни от какой другой), частная производная по конкретной координате считается очень просто — это обычная производная `l` по её второму аргументу, взятая в точке `F(x_i)`:

In [ ]:
∂L/∂F(x_i) = ∂l(y_i, F(x_i)) / ∂F(x_i)

**Антиградиент** в этом `N`-мерном пространстве — это вектор, компоненты которого:

In [ ]:
r_i = - ∂l(y_i, F(x_i)) / ∂F(x_i)      (для i = 1, ..., N)

Величины `r_i` называются **псевдо-остатками (pseudo-residuals)**. Это буквально «на сколько и в какую сторону нужно сдвинуть предсказание для объекта `i`, чтобы уменьшить его вклад в общий loss» — прямое применение той же логики антиградиента из раздела 3, только теперь у нас `N` «переменных» вместо `k` весов.

**А теперь — главная проблема.** Если бы мы остановились здесь и просто сделали `F(x_i)_new = F(x_i)_old + η·r_i` для каждого обучающего объекта, мы бы улучшили предсказания **только для тех `N` точек, что уже есть в обучающей выборке**. У нас не было бы вообще никакого способа посчитать предсказание для **нового**, ранее не виденного объекта `x_new` — ведь `r_i` мы считали как отдельное число для каждой конкретной обучающей точки, никакой «формулы», применимой к произвольному `x`, у нас нет.

**Решение — гениальная по простоте идея гradient boosting:** обучим **дополнительную модель** `h(x)` (в градиентном бустинге — регрессионное дерево, Модуль 4) **приближать** зависимость между `x_i` и соответствующим псевдо-остатком `r_i`, как будто это обычная задача регрессии с признаками `x_i` и целевой переменной `r_i`. После этого `h(x)` можно применить **к любому** `x`, включая невиданные объекты — мы получили **обобщённое (generalized) приближение направления антиградиента**, а не просто набор из `N` чисел.

Обновление модели:

In [ ]:
F_new(x) = F_old(x) + η · h(x)

Это и называется **градиентным спуском в пространстве функций (functional gradient descent):** мы по-прежнему двигаемся против антиградиента функции потерь, но каждый «шаг» — это не число и не вектор фиксированной длины, а **целая обучаемая функция**, которая аппроксимирует антиградиент и умеет обобщаться на новые данные. Именно поэтому в градиентном бустинге всегда есть промежуточный шаг «обучить дерево на остатках» — это буквально попытка выучить функцию, аппроксимирующую антиградиент.

## 5. Вывод псевдо-остатка для MSE

Пусть `l(y, F) = (1/2)(y - F)²` (множитель `1/2` — для удобства производной).

**Дифференцируем по `F`** (используем правило дифференцирования сложной функции — цепное правило):

In [ ]:
∂l/∂F = 2 · (1/2) · (y - F) · (-1) = -(y - F) = F - y

**Антиградиент:**

In [ ]:
r = -∂l/∂F = y - F

**Вывод:** для MSE антиградиент **в точности совпадает** с обычным остатком `y - F`, который вы уже видели в линейной регрессии. Отсюда и родилась народная формулировка «в градиентном бустинге каждое следующее дерево обучается на остатках предыдущего» — это буквально верно, но **только для MSE**. Для других функций потерь это уже не совсем «остаток» в привычном смысле — следующий раздел показывает, почему.

## 6. Вывод псевдо-остатка для LogLoss (бинарная классификация)

Здесь нужно быть внимательнее: `F(x)` в классификации — это не вероятность, а **сырой счёт (raw score)**, часто называемый **логитом** или **лог-оддсом** — произвольное вещественное число, которое **преобразуется в вероятность** через сигмоиду (уже знакомую вам с Недели 4):

In [ ]:
p = σ(F) = 1 / (1 + e^(-F))

Функция потерь LogLoss записывается через вероятность `p`, а не напрямую через `F`:

In [ ]:
l(y, F) = -[ y·log(p) + (1-y)·log(1-p) ],   где p = σ(F)

Чтобы взять `∂l/∂F`, нужна **цепочка** производных: `l` зависит от `p`, а `p` зависит от `F`. По цепному правилу:

In [ ]:
∂l/∂F = (∂l/∂p) · (∂p/∂F)

**Шаг 1 — считаем `∂l/∂p`:**

In [ ]:
∂l/∂p = -y/p + (1-y)/(1-p)

**Шаг 2 — считаем `∂p/∂F`.** Это стандартная производная сигмоиды, которая элегантно выражается через саму сигмоиду:

In [ ]:
∂p/∂F = p·(1-p)

*(Если хотите проверить самостоятельно: `p = 1/(1+e^{-F})`, по правилу производной частного и цепному правилу получается ровно `p(1-p)` — этот результат стоит просто запомнить, он встречается очень часто во всём, что связано с логистической регрессией и нейросетями.)*

**Шаг 3 — перемножаем и упрощаем:**

In [ ]:
∂l/∂F = [ -y/p + (1-y)/(1-p) ] · p(1-p)
       = -y·(1-p) + (1-y)·p
       = -y + yp + p - yp
       = p - y

Обратите внимание, как красиво всё сократилось — оба слагаемых с `yp` взаимно уничтожились. Это не случайность, а следствие того, что сигмоида — **канонический (natural) link** именно для LogLoss в терминах обобщённых линейных моделей: при таком удачном сочетании функции потерь и функции связи производная всегда сворачивается в форму «предсказание минус факт».

**Итог:**

In [ ]:
∂l/∂F = p - y      ->      антиградиент r = -(p - y) = y - p

**Сравнение с MSE:** формально антиградиент снова выглядит как «истина минус предсказание» — `y - p`. Но есть принципиальная разница, которая и является ответом на первый чек-поинт вопрос модуля: в MSE `y` и `F` — величины **одной природы** (обе — прямые значения таргета), а в LogLoss остаток берётся между `y` (метка 0/1) и `p = σ(F)` — то есть между меткой и **вероятностью**, полученной после нелинейного преобразования сырого счёта `F`. Само `F` (то, что предсказывает ансамбль деревьев и что реально обновляется на каждом шаге) находится в **другой шкале** (логитов, а не вероятностей) — и остаток `y - p` относится не к `F` напрямую, а к его образу после сигмоиды. Совпадение по форме формулы — приятное свойство канонического link'а, а не признак того, что механизм идентичен MSE-случаю.

## 7. Численный пример: считаем псевдо-остатки руками

### 7.1. Регрессия (MSE)

Игрушечные данные (например, суммы возврата средств по 5 обращениям): `y = [3, 5, 2, 8, 6]`.

**Инициализация `F_0`.** Оптимальная **константа**, минимизирующая MSE по всей выборке — это среднее значение `y` (можно показать через `d/dc [Σ(y_i - c)²] = 0 -> c = mean(y)`, классический результат):

In [ ]:
F_0 = mean(y) = (3+5+2+8+6)/5 = 24/5 = 4.8

**Псевдо-остатки** `r_i = y_i - F_0`:

| i | y_i | F_0 | r_i = y_i - F_0 |
|---|-----|-----|------------------|
| 1 | 3 | 4.8 | -1.8 |
| 2 | 5 | 4.8 | 0.2 |
| 3 | 2 | 4.8 | -2.8 |
| 4 | 8 | 4.8 | 3.2 |
| 5 | 6 | 4.8 | 1.2 |

Эти пять чисел (`-1.8, 0.2, -2.8, 3.2, 1.2`) и станут **целевой переменной** для обучения первого дерева `h_1(x)` в Модуле 4 — дерево будет решать обычную задачу регрессии, стараясь предсказать именно эти значения по признакам `x`.

### 7.2. Классификация (LogLoss) — на нашем антифрод-датасете из Модуля 1

Используем 8 транзакций из Модуля 1: 4 мошеннические (`y=1`), 4 честные (`y=0`).

**Инициализация `F_0`.** Оптимальная константа для LogLoss — это **лог-оддс** общей доли положительного класса:

In [ ]:
p_prior = 4/8 = 0.5
F_0 = ln( p_prior / (1 - p_prior) ) = ln(0.5/0.5) = ln(1) = 0

*(Проверка на менее сбалансированном примере, как в реальном антифроде: если бы `p_prior = 0.1` (1 мошенничество на 10 транзакций), `F_0 = ln(0.1/0.9) = ln(0.111) ≈ -2.197` — сильно отрицательный логит, что соответствует низкой базовой вероятности мошенничества, интуитивно верно.)*

**Переводим `F_0` обратно в вероятность:** `p_0 = σ(F_0) = σ(0) = 1/(1+e^0) = 1/2 = 0.5` — то есть до обучения первого дерева модель предсказывает **одинаковую** вероятность 0.5 для вообще любого объекта (константа, не зависящая от признаков — логично, ведь `F_0` не использует `x` вообще).

**Псевдо-остатки** `r_i = y_i - p_0`:

| i | hour | y_i (fraud) | p_0 | r_i = y_i - p_0 |
|---|------|-------------|-----|------------------|
| 1 | 1  | 1 | 0.5 | **+0.5** |
| 2 | 2  | 1 | 0.5 | **+0.5** |
| 3 | 3  | 1 | 0.5 | **+0.5** |
| 4 | 9  | 0 | 0.5 | **-0.5** |
| 5 | 10 | 0 | 0.5 | **-0.5** |
| 6 | 11 | 0 | 0.5 | **-0.5** |
| 7 | 14 | 0 | 0.5 | **-0.5** |
| 8 | 23 | 1 | 0.5 | **+0.5** |

**Интерпретация, которая должна «щёлкнуть»:** каждый мошеннический объект получил псевдо-остаток `+0.5` («модель должна увеличить логит для этого объекта»), каждый честный — `-0.5` («модель должна уменьшить логит»). Первое дерево в Модуле 4 будет обучаться **именно на этих значениях** как на целевой переменной обычной регрессии — и, что приятно, мы уже знаем из Модуля 1, что признак `hour ≤ 6` идеально разделяет часть объектов по классам, значит дерево сможет почти идеально предсказать эти `+0.5`/`-0.5` для соответствующей группы точек.

## 8. Общий алгоритм (пока без деталей реализации — они в Модуле 4)

Теперь можно формально записать общую схему, которая будет наполняться конкретикой весь оставшийся курс:

1. Инициализировать `F_0(x) = argmin(c) Σ l(y_i, c)` — константа, оптимальная по всей выборке (среднее для MSE, лог-оддс доли класса для LogLoss).
2. **Для `m = 1, ..., M`:**
   a. Посчитать псевдо-остатки: `r_i = -∂l(y_i, F)/∂F` при `F = F_{m-1}(x_i)`, для каждого `i`.
   b. Обучить **регрессионное** дерево `h_m(x)`, приближающее зависимость `x -> r`.
   c. Обновить модель: `F_m(x) = F_{m-1}(x) + η · h_m(x)`.
3. Итоговая модель: `F_M(x)` (для классификации — дополнительно применяем `σ(·)`, чтобы получить вероятность).

Обратите внимание на пункт (b): дерево **всегда** решает задачу регрессии (предсказывает вещественное число — псевдо-остаток), даже если исходная задача курса — бинарная классификация. Это частый источник путаницы: «почему LightGBM с `objective='binary'` внутри всё равно строит регрессионные деревья?» — потому что предсказывается не класс, а величина антиградиента, которая всегда непрерывна.

## 9. Практика: код

### 9.1. Псевдо-остатки для MSE — сверка формулы с прямым вычислением

In [ ]:
import numpy as np

y = np.array([3, 5, 2, 8, 6], dtype=float)

# Инициализация F_0 - среднее
F0 = y.mean()
print("F0 =", F0)

# Псевдо-остатки по формуле производной MSE: r = y - F
pseudo_residuals = y - F0
print("Псевдо-остатки (по формуле градиента):", pseudo_residuals)

# Прямая проверка: обычный остаток y_true - y_pred, где y_pred = F0 для всех
plain_residuals = y - np.full_like(y, F0)
print("Обычные остатки:", plain_residuals)

assert np.allclose(pseudo_residuals, plain_residuals)
print("Совпадают -> для MSE псевдо-остаток буквально равен обычному остатку")

### 9.2. Псевдо-остатки для LogLoss — на игрушечном антифрод-датасете

In [ ]:
import numpy as np

hour = np.array([1, 2, 3, 9, 10, 11, 14, 23])
y = np.array([1, 1, 1, 0, 0, 0, 0, 1])

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Инициализация F0 - лог-оддс доли положительного класса
p_prior = y.mean()
F0 = np.log(p_prior / (1 - p_prior))
print("p_prior =", p_prior, " F0 (логит) =", F0)

p0 = sigmoid(F0)
print("p0 (вероятность из F0) =", p0)  # должно быть точно равно p_prior

# Псевдо-остатки: r = y - p
pseudo_residuals = y - p0
print("Псевдо-остатки:", pseudo_residuals)

Убедитесь, что запустив этот код, вы получите ровно те же числа `+0.5`/`-0.5`, что и в ручном расчёте раздела 7.2 — это хорошая привычка при изучении матчасти: всегда сверять аналитический вывод с прямым вычислением в коде.

### 9.3. Проверка производной сигмоиды численно (для скептиков)

Если хочется дополнительно убедиться, что `dp/dF = p(1-p)` — можно проверить это **численно**, через определение производной как предела:

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

F = 0.7
eps = 1e-6

numeric_derivative = (sigmoid(F + eps) - sigmoid(F - eps)) / (2 * eps)
analytic_derivative = sigmoid(F) * (1 - sigmoid(F))

print("Численная производная:", numeric_derivative)
print("Аналитическая формула p(1-p):", analytic_derivative)

Оба значения должны совпасть с точностью до нескольких знаков после запятой — это универсальный приём проверки любой производной, которую вы вывели руками, полезно держать в арсенале на весь курс.

## 10. Частые вопросы на собеседовании

| Вопрос | На что обратить внимание в ответе |
|---|---|
| Почему следующее дерево в бустинге обучается именно на остатках/псевдо-остатках? | Это приближённая реализация шага градиентного спуска: остаток — это (анти)градиент функции потерь по предсказанию; обучая дерево на нём, мы обучаем функцию, аппроксимирующую направление наискорейшего убывания ошибки |
| Почему нельзя просто «подвинуть» предсказания на антиградиент напрямую? | Это сработало бы только для объектов, которые уже есть в обучающей выборке — не даёт способа предсказывать для новых данных; дерево нужно, чтобы **обобщить** направление антиградиента как функцию от `x` |
| Почему слабый learner в градиентном бустинге всегда решает задачу регрессии? | Псевдо-остаток — всегда вещественное число (антиградиент loss по предсказанию), вне зависимости от того, регрессия или классификация исходная задача |
| Чем псевдо-остаток для LogLoss отличается от обычного остатка? | Формально имеет ту же форму «истина минус предсказание», но берётся между меткой `y` и **вероятностью** `p=σ(F)`, а не напрямую между `y` и сырым логитом `F`, который реально обновляется |
| Что такое `F_0` и почему для классификации это не 0.5? | `F_0` — оптимальная константа в шкале **логитов** (не вероятностей), минимизирующая loss по всей выборке; для сбалансированных классов `F_0=0` (что соответствует `p=0.5`), но при дисбалансе `F_0` — это лог-оддс реальной доли класса, а не произвольные 0.5 |

## 11. Чек-поинт — попробуйте ответить без подсказок

1. Почему для MSE псевдо-остаток совпадает с обычным остатком, а для LogLoss — нет?
2. Что значит фраза «градиентный спуск в пространстве функций»?
3. Почему нельзя просто взять антиградиент и напрямую обновить предсказания в обучающих точках, не обучая дерево?
4. Почему слабый learner в бустинге всегда решает задачу регрессии, даже если исходная задача — классификация?
5. Как вычисляется начальное приближение `F_0`, и почему для LogLoss это не 0.5, а лог-оддс?

## Ответы для самопроверки

<details>
<summary>Раскрыть после того, как попробуете ответить сами</summary>

1. Для MSE производная `∂l/∂F` даёт ровно `F - y`, поэтому антиградиент `y - F` — это буквально та же величина, что и «истина минус предсказание» в привычном смысле линейной регрессии, где `y` и `F` находятся в одной и той же шкале (единицах измерения таргета). Для LogLoss `F` — это логит (сырой счёт в шкале логарифма отношения шансов), а не вероятность; антиградиент `y - p` берётся между меткой `y` и **вероятностью** `p = σ(F)` — производной величиной, полученной нелинейным преобразованием `F`. Совпадение по форме («истина минус что-то») — следствие удачного сочетания LogLoss и сигмоиды (канонический link), а не признак того, что механизм идентичен MSE.

2. Обычный градиентный спуск оптимизирует фиксированный вектор параметров, двигая его против градиента. В функциональном градиентном спуске мы временно рассматриваем **вектор предсказаний на обучающих точках** `(F(x_1),...,F(x_N))` как «параметры», считаем антиградиент функции потерь по этому вектору (это и есть псевдо-остатки), а затем **обучаем модель (дерево)**, чтобы она приближала эту антиградиентную функцию как зависимость от `x` — то есть шагом градиентного спуска становится не число и не фиксированный вектор, а **целая обучаемая функция**, обобщающая направление убывания ошибки на любые, в том числе новые, объекты.

3. Потому что антиградиент, посчитанный напрямую, — это просто набор из `N` чисел, привязанных к конкретным `N` обучающим объектам. У нас нет никакой «формулы», которая говорила бы, что делать с произвольным новым `x_new`, не входящим в этот список. Обучая дерево `h(x)` приближать зависимость между `x` и антиградиентом, мы получаем функцию, которую можно вычислить для **любого** `x` — это и обеспечивает обобщающую способность модели на новых данных.

4. Псевдо-остаток по определению — это значение производной функции потерь по предсказанию, взятое с обратным знаком; производная — всегда вещественное число (может быть любым, положительным или отрицательным, любой величины), независимо от того, что представляет собой исходная переменная `y` (метка класса или непрерывное число). Поэтому задача «предсказать псевдо-остаток по признакам `x`» — всегда задача регрессии, даже когда конечная цель всего пайплайна — классификация.

5. `F_0` определяется как константа `c`, минимизирующая суммарный loss по всей обучающей выборке: `F_0 = argmin(c) Σ l(y_i, c)`. Для MSE решение этой минимизации — среднее арифметическое `y` (в шкале самого таргета). Для LogLoss минимизация даёт **лог-оддс** доли положительного класса: `F_0 = ln(p/(1-p))`, где `p` — доля класса 1 в обучающей выборке. Это не 0.5 (и не `p` напрямую), потому что `F_0` живёт в шкале логитов, а не вероятностей — 0.5 в шкале вероятностей действительно соответствует `F_0=0`, но только в частном случае идеально сбалансированных классов (`p=0.5`); при любом дисбалансе `F_0` сдвигается в сторону, соответствующую реальной базовой частоте класса.

</details>